In [ ]:
# ==============================
#  MOVIE RECOMMENDATION SYSTEM
# ==============================

# Install required libraries
!pip install pandas numpy scikit-learn ipywidgets

# Imports
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MultiLabelBinarizer
import ipywidgets as widgets
from IPython.display import display

# ==============================
#  LOAD DATA (MovieLens small)
# ==============================

# Download dataset
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

# Load data
movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")

# Preview
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)

# ==============================
#  DATA PREPROCESSING
# ==============================

# Clean genres
movies['genres'] = movies['genres'].apply(lambda x: x.split('|'))

# Convert genres into binary features
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genres'])

genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_)
movies = pd.concat([movies, genre_df], axis=1)

# Merge ratings with movies
data = pd.merge(ratings, movies, on='movieId')

# ==============================
# COLLABORATIVE FILTERING
# ==============================

# Create user-item matrix
user_item_matrix = data.pivot_table(index='userId', columns='title', values='rating').fillna(0)

# Apply SVD
svd = TruncatedSVD(n_components=20, random_state=42)
matrix_reduced = svd.fit_transform(user_item_matrix)

# Reconstruct approximate matrix
matrix_reconstructed = np.dot(matrix_reduced, svd.components_)

collab_df = pd.DataFrame(matrix_reconstructed, columns=user_item_matrix.columns)

# ==============================
#  CONTENT-BASED FILTERING
# ==============================

# Compute similarity based on genres
cosine_sim = cosine_similarity(genre_df)

# Map indices
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def content_recommend(title, n=5):
    if title not in indices:
        return []

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:n+1]
    movie_indices = [i[0] for i in sim_scores]

    return movies['title'].iloc[movie_indices].tolist()

# ==============================
#  HYBRID RECOMMENDER
# ==============================

def hybrid_recommend(user_id, favorite_movie, n=5):

    # Collaborative filtering recommendations
    if user_id in collab_df.index:
        user_scores = collab_df.loc[user_id]
        collab_recs = user_scores.sort_values(ascending=False).head(n*2).index.tolist()
    else:
        collab_recs = []

    # Content-based recommendations
    content_recs = content_recommend(favorite_movie, n*2)

    # Combine results
    combined = list(dict.fromkeys(collab_recs + content_recs))

    return combined[:n]

# ==============================
#  USER INTERFACE (COLAB)
# ==============================

# Widgets
user_input = widgets.IntText(value=1, description='User ID:')
movie_input = widgets.Text(value='Toy Story (1995)', description='Favorite Movie:')
button = widgets.Button(description='Recommend')
output = widgets.Output()

def on_button_click(b):
    output.clear_output()

    user_id = user_input.value
    movie_name = movie_input.value

    recs = hybrid_recommend(user_id, movie_name, n=5)

    with output:
        print(" Top 5 Recommendations:\n")
        for i, movie in enumerate(recs, 1):
            print(f"{i}. {movie}")

button.on_click(on_button_click)

# Display UI
display(user_input, movie_input, button, output)

# ==============================
#  EXAMPLE RUN (OPTIONAL)
# ==============================

print("\nExample Recommendations:")
print(hybrid_recommend(1, "Toy Story (1995)"))